# V1-S15 — F1 LEVELS diagnostic (pre-registered PR3D, computed ONCE)

This notebook runs the **once-only** F1 *levels* diagnostic locked in
`docs/preregistrations/PR3D_levels_diagnostic.md` (committed `7d47a21`, BEFORE this compute —
the anti-HARKing ordering `commit(PR3D) precedes commit(notebook-15 results)`). It characterizes
whether F1's signed differenced null (Gate G5, 2026-06-08) is a **TRUE** level-null, an
**APPARENT** one (the once-differenced test was blind to a level/cointegration relationship), or
**AMBIGUOUS** — by re-running the SAME pre-registered machinery on the **undifferenced LEVEL**
series via the Toda-Yamamoto (TY) augmented-VAR Granger test, plus an Engle-Granger ECM
robustness characterization and a decisive power simulation.

**This is a DIAGNOSTIC, not a finding — it CANNOT flip Gate G5** (the verdict JSON carries
`does_not_flip_g5: true`). A flip would require a separate confirmatory PR4 + additive G5'
+ Samer's Phase-2 checkpoint (PR3D §8.2). The signed G5 and parent PR3 are NEVER edited here.

All statistics live in the **pure** modules `scifield.findings.cascade` (TY + Engle-Granger ECM
+ level-panel, committed `c43ff65`) and `scifield.findings.cascade_powersim`; this notebook only
does parquet I/O, the `epistemic_extracted ⋈ novelty_semantic` join, and applies the PR3D §8.1
verdict table. **$0, CPU-only, no network/GPU/DeepSeek.** Reported symmetrically: a TRUE or
AMBIGUOUS verdict is as reportable as an APPARENT one (PR3D §8, §9).

**Universe (identical to PR3 / notebook 11):** `epistemic_extracted.parquet`
(`pmid, study_design`) INNER-joined to `novelty_semantic.parquet` (`pmid, topic_id, year`) on
`pmid`, LEAF topics only (`topic_id != -1`) — ≈ 69,339 papers across 149 leaf topics, of which
**138 qualify** (the fixed holds-fraction denominator). Primary quality = `mean_tier`;
robustness = `rct_share` (PR3D §8.3); the RCT-share rerun cannot convert a null primary to a hold.

## 1. Setup

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.lines import Line2D  # noqa: E402

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

from scifield.findings import cascade as C  # noqa: E402
from scifield.findings import cascade_powersim as PS  # noqa: E402
from scifield.repro import record_run  # noqa: E402

# Pinned PR3 / PR3D parameters (echoed for sidecars; these ARE the module defaults).
PR3D_PARAMS = {
    "tier_map": dict(C.TIER_MAP),  # RCT=4 / cohort=3 / case_control=2 / case_series=1
    "v_min": 30,
    "min_years": 8,
    "min_count": 5,
    "max_nan_frac": 0.25,
    "ty_k": 3,
    "ty_d_max": 1,
    "ty_var_order_p": 4,
    "ty_trend": "c",
    "ty_wald_df": 3,
    "granger_lag": 3,
    "fdr_q": 0.05,
    "holds_frac": 0.20,
    "panel_alpha": 0.05,
    "sign_convention": "negative CCF lag / quality->volume = quality LEADS",
}

# Power-sim locked grid (PR3D §7.2) — the cascade_powersim defaults, echoed.
POWERSIM_GRID = {
    "T_grid": [8, 15, 22, 30, 40],
    "beta_grid": [0.0, 0.25, 0.5, 1.0, 2.0],
    "dgps": ["null", "level_cascade", "diff_cascade", "level_cascade_coint"],
    "n_rep": 500,
    "lag": 2,
    "alpha": 0.05,
    "seed": 20260609,
    "verdict_power_read_dgp": "level_cascade_coint",
    "verdict_power_read_T": 30,
}


def _load_parquet(path: Path, columns=None) -> pd.DataFrame | None:
    """Defensive parquet read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING artifact: {path} — cannot run the analysis.")
        return None
    return pd.read_parquet(path, columns=columns)


print("matplotlib:", matplotlib.__version__, "| backend:", matplotlib.get_backend())
print("DATA       :", DATA)
print("FIGURES_DIR:", FIGURES_DIR)

matplotlib: 3.10.9 | backend: Agg
DATA       : /Users/samersalman/Desktop/SciField/data/v1
FIGURES_DIR: /Users/samersalman/Desktop/SciField/docs/figures


## 2. I/O + join — assemble the F1 universe (identical to PR3 / notebook 11 §4)

INNER-join `epistemic_extracted[pmid, study_design]` to `novelty_semantic[pmid, topic_id, year]`
on `pmid`; keep LEAF topics only (`topic_id != -1`). The module is pure: it receives this
already-joined, already-leaf-filtered tidy frame and does the rest. This block is byte-identical
in intent to notebook 11 cell 2, so the 138-topic denominator is wired the same way.

In [2]:
EPISTEMIC = DATA / "epistemic_extracted.parquet"
NOVELTY = DATA / "novelty_semantic.parquet"

ep = _load_parquet(EPISTEMIC, columns=["pmid", "study_design"])
nov = _load_parquet(NOVELTY, columns=["pmid", "topic_id", "year"])
assert ep is not None and nov is not None, "F1 inputs missing — cannot proceed."

df = ep.merge(nov, on="pmid", how="inner")  # INNER join on pmid
n_joined = len(df)
df = df[df["topic_id"] != -1].copy()  # LEAF topics only
n_leaf = len(df)
n_leaf_topics = int(df["topic_id"].nunique())

designs = sorted(df["study_design"].dropna().unique().tolist())
print(f"epistemic rows = {len(ep):,} | novelty rows = {len(nov):,}")
print(f"INNER join     = {n_joined:,} rows")
print(f"leaf-filtered  = {n_leaf:,} papers across {n_leaf_topics} leaf topics (F1 universe)")
print(f"year span      = {int(df['year'].min())}–{int(df['year'].max())}")
print(f"study_design values = {designs}")
# Sanity vs PR3 §4 pre-lock denominators (≈ 69,339 papers, 149 leaf topics).
assert set(designs) <= {
    "RCT",
    "cohort",
    "case_control",
    "case_series",
    "review",
    "other",
}, "unexpected study_design label — does not match TIER_MAP universe"

epistemic rows = 91,230 | novelty rows = 89,230
INNER join     = 91,230 rows
leaf-filtered  = 69,339 papers across 149 leaf topics (F1 universe)
year span      = 1995–2026
study_design values = ['RCT', 'case_control', 'case_series', 'cohort', 'other', 'review']


## 3. The LEVELS analysis loop (PR3D §§5-6, reusing PR3 machinery verbatim)

Mirrors notebook 11's `run_f1`, with the **one deliberate, pre-declared difference**: it feeds the
**undifferenced LEVEL** series from `prepare_series` DIRECTLY to `toda_yamamoto_pair` (NOT
`np.diff`). For a qualifying topic with `status == "ok"` it runs the bidirectional TY Wald, pools
**both** p-values into one vector, collects the LEVEL pair for the level-panel, and runs the
Engle-Granger ECM characterization. BH-FDR is applied **once over the whole pool**;
`classify_direction` sets each topic's direction; `panel_toda_yamamoto` runs on the level pairs;
`decide_f1(n_qualifying=138, holds_frac=0.20, panel_alpha=0.05)` returns the HOLD/NULL verdict.
Non-evaluable / too-short topics STAY in the 138 denominator as `direction="none"` (conservative,
PR3 §5) and are added to neither the pool nor the panel.

In [3]:
def run_f1_levels(frame: pd.DataFrame, *, quality_col: str) -> dict:
    """Execute the full pre-registered F1 LEVELS loop for one quality series.

    Clones notebook 11 `run_f1` but feeds the UNDIFFERENCED level series to the
    Toda-Yamamoto test (`toda_yamamoto_pair`), collects level pairs for the level
    panel (`panel_toda_yamamoto`), and runs the Engle-Granger ECM robustness per
    evaluable topic. Calls the pure `cascade` helpers only — never reimplements them.
    """
    series = C.build_topic_year_series(frame)  # one row per (topic, year)
    topics = C.qualifying_topics(series)  # defaults 30/8/5 -> the 138 denominator
    n_qualifying = len(topics)

    pooled_pvals: list[float] = []  # ALL per-topic per-direction TY p's, one flat vector
    pool_index: list[tuple] = []  # parallel: pool_position -> (topic_id, direction)
    level_pairs: list[tuple] = []  # (quality_levels, volume_levels) per evaluable topic
    per_topic: list[dict] = []  # per-topic results table rows
    n_non_evaluable = 0
    n_too_short = 0

    for topic in topics:
        topic_series = series[series.topic_id == topic]
        quality, volume, status = C.prepare_series(topic_series, quality_col=quality_col)

        if status == "ok":
            # LEVEL series fed DIRECTLY to TY — NOT differenced (the whole point).
            p_qv, p_vq = C.toda_yamamoto_pair(quality, volume)
            pos_qv = len(pooled_pvals)
            pooled_pvals.append(p_qv)
            pool_index.append((int(topic), "q_to_v"))
            pos_vq = len(pooled_pvals)
            pooled_pvals.append(p_vq)
            pool_index.append((int(topic), "v_to_q"))
            level_pairs.append((quality.to_numpy(), volume.to_numpy()))
            ecm = C.engle_granger_ecm_pair(quality, volume)
            per_topic.append(
                {
                    "topic_id": int(topic),
                    "status": status,
                    "series_len": int(len(quality)),
                    "p_q_to_v": float(p_qv),
                    "p_v_to_q": float(p_vq),
                    "leads_sig": False,  # filled after pooled FDR
                    "lags_sig": False,
                    "direction": "none",
                    "coint_pvalue": float(ecm["coint_pvalue"]),
                    "cointegrated": bool(ecm["cointegrated"]),
                    "ec_coef": float(ecm["ec_coef"]),
                    "ec_pvalue": float(ecm["ec_pvalue"]),
                    "_pos_qv": pos_qv,
                    "_pos_vq": pos_vq,
                }
            )
        else:
            # non_evaluable / too_short STAY in the 138 denominator as direction='none';
            # NOT added to the pool or the panel (conservative — PR3 §5 / PR3D §5).
            if status == "non_evaluable":
                n_non_evaluable += 1
            elif status == "too_short":
                n_too_short += 1
            per_topic.append(
                {
                    "topic_id": int(topic),
                    "status": status,
                    "series_len": 0,
                    "p_q_to_v": float("nan"),
                    "p_v_to_q": float("nan"),
                    "leads_sig": False,
                    "lags_sig": False,
                    "direction": "none",
                    "coint_pvalue": float("nan"),
                    "cointegrated": False,
                    "ec_coef": float("nan"),
                    "ec_pvalue": float("nan"),
                    "_pos_qv": None,
                    "_pos_vq": None,
                }
            )

    # THE multiple-comparison correction: BH-FDR over the WHOLE pool (all topics, both
    # directions). NaN p's come back not-rejected and do not inflate m.
    reject = C.bh_fdr(np.array(pooled_pvals, dtype="float64"))
    for row in per_topic:
        if row["_pos_qv"] is not None:
            leads_sig = bool(reject[row["_pos_qv"]])  # its q_to_v reject flag
            lags_sig = bool(reject[row["_pos_vq"]])  # its v_to_q reject flag
            row["leads_sig"] = leads_sig
            row["lags_sig"] = lags_sig
            row["direction"] = C.classify_direction(leads_sig=leads_sig, lags_sig=lags_sig)

    panel = C.panel_toda_yamamoto(level_pairs)  # (p_q_to_v, p_v_to_q) level block-F
    verdict = C.decide_f1(per_topic, panel, n_qualifying=n_qualifying)

    results_df = pd.DataFrame(
        [{k: v for k, v in row.items() if not k.startswith("_")} for row in per_topic]
    )
    return {
        "verdict": verdict,
        "results_df": results_df,
        "panel": (float(panel[0]), float(panel[1])),
        "n_qualifying": int(n_qualifying),
        "n_non_evaluable": int(n_non_evaluable),
        "n_too_short": int(n_too_short),
        "n_evaluable": int(len(level_pairs)),
        "pool_size": int(len(pooled_pvals)),
        "series": series,
    }

### 3.1 PRIMARY levels run — mean-tier quality series

In [4]:
primary = run_f1_levels(df, quality_col="mean_tier")
vP = primary["verdict"]

print("=== LEVELS PRIMARY (mean-tier, Toda-Yamamoto) ===")
print(f"n_qualifying      = {primary['n_qualifying']}  (PR3 fixed denominator = 138)")
print(f"n_non_evaluable   = {primary['n_non_evaluable']}   (>25% NaN-tier; kept in denom)")
print(f"n_too_short       = {primary['n_too_short']}")
print(f"n_evaluable       = {primary['n_evaluable']}   (pooled p-values = {primary['pool_size']})")
print(
    f"direction split   : lead={vP['n_lead']}  lag={vP['n_lag']}  "
    f"coupled={vP['n_coupled']}  none={vP['n_none']}"
)
print(
    f"n_directional     = {vP['n_directional']}   frac = {vP['frac_directional']:.4f}  "
    f"(holds bar = {vP['frac_threshold']:.2f} -> >= 28/138)"
)
print(f"dominant_direction= {vP['dominant_direction']}   panel_agrees = {vP['panel_agrees']}")
print(
    f"panel TY p: quality->volume = {vP['panel_p_quality_leads']:.4g}  "
    f"volume->quality = {vP['panel_p_quality_lags']:.4g}"
)
print(f"\nLEVELS TEST HOLDS (primary mean_tier) = {vP['holds']}")

=== LEVELS PRIMARY (mean-tier, Toda-Yamamoto) ===
n_qualifying      = 138  (PR3 fixed denominator = 138)
n_non_evaluable   = 10   (>25% NaN-tier; kept in denom)
n_too_short       = 0
n_evaluable       = 128   (pooled p-values = 256)
direction split   : lead=5  lag=2  coupled=0  none=131
n_directional     = 7   frac = 0.0507  (holds bar = 0.20 -> >= 28/138)
dominant_direction= quality_leads   panel_agrees = False
panel TY p: quality->volume = 0.07415  volume->quality = 0.8996

LEVELS TEST HOLDS (primary mean_tier) = False


### 3.2 Honesty check (a) — series-length (T) distribution of the directional topics

PR3D integrity requirement: if the levels test holds, a "hold" concentrated in **short-T** topics
(where the TY χ² Wald over-rejects, §7.3) is likely size inflation, NOT signal. Here we report the
T distribution of any FDR-significant directional topics against the evaluable-topic median, so a
reader can judge whether the directional signal sits at lengths where TY controls size (T >= 30).

In [5]:
rdfP = primary["results_df"]
ok_mask = rdfP["status"] == "ok"
T_eval = rdfP.loc[ok_mask, "series_len"].to_numpy()
directional_mask = rdfP["direction"].isin(["lead", "lag"])
dir_rows = rdfP[directional_mask].copy()

T_eval_median = int(np.median(T_eval)) if T_eval.size else 0
T_eval_min = int(T_eval.min()) if T_eval.size else 0
T_eval_max = int(T_eval.max()) if T_eval.size else 0
print(f"Evaluable-topic T (series length): n={T_eval.size}  median={T_eval_median}")
print(f"  T range: min={T_eval_min}  max={T_eval_max}")
print(f"\nFDR-significant DIRECTIONAL topics: n={len(dir_rows)}")
if len(dir_rows):
    T_dir = dir_rows["series_len"].to_numpy()
    n_short = int((T_dir < 30).sum())
    cols_show = ["topic_id", "series_len", "direction", "p_q_to_v", "p_v_to_q"]
    print(dir_rows[cols_show].to_string(index=False))
    print(
        f"\n  directional-topic T: median={int(np.median(T_dir))}  "
        f"min={int(T_dir.min())}  max={int(T_dir.max())}  | {n_short}/{len(dir_rows)} have T<30"
    )
    SHORT_T_FLAG = bool(n_short > len(dir_rows) / 2)
    if SHORT_T_FLAG:
        msg = (
            "CONCENTRATED in short-T (T<30) topics where TY over-rejects "
            "-> likely size inflation, not signal."
        )
    else:
        msg = "NOT concentrated in short-T topics (most holders T>=30, where TY controls size)."
    print("  HONESTY FLAG (a): directional signal is " + msg)
else:
    SHORT_T_FLAG = False
    print("  (no directional topics — honesty check (a) is moot; the levels test is null on HL1.)")

Evaluable-topic T (series length): n=128  median=31
  T range: min=9  max=32

FDR-significant DIRECTIONAL topics: n=7
 topic_id  series_len direction  p_q_to_v  p_v_to_q
        6          32      lead  0.000316  0.384863
       24          32      lead  0.000046  0.683752
       58          28      lead  0.000115  0.390719
       76          29       lag  0.190402  0.000839
      110          29       lag  0.894896  0.000122
      134          15      lead  0.000041  0.727102
      136          19      lead  0.001303  0.481189

  directional-topic T: median=29  min=15  max=32  | 5/7 have T<30
  HONESTY FLAG (a): directional signal is CONCENTRATED in short-T (T<30) topics where TY over-rejects -> likely size inflation, not signal.


### 3.3 ROBUSTNESS levels rerun — RCT-share quality series (PR3D §8.3)

Identical levels loop with `quality_col="rct_share"`. `rct_share` has no structural NaNs, so no
topic should be non-evaluable here. The RCT-share verdict characterizes robustness and **cannot**
convert a null primary into a hold (PR3D §9).

In [6]:
rct = run_f1_levels(df, quality_col="rct_share")
vR = rct["verdict"]

print("=== LEVELS ROBUSTNESS (rct_share, Toda-Yamamoto) ===")
print(f"n_qualifying      = {rct['n_qualifying']}")
print(f"n_non_evaluable   = {rct['n_non_evaluable']}   (expected 0; gap-free series)")
print(f"n_evaluable       = {rct['n_evaluable']}   (pooled p-values = {rct['pool_size']})")
print(
    f"direction split   : lead={vR['n_lead']}  lag={vR['n_lag']}  "
    f"coupled={vR['n_coupled']}  none={vR['n_none']}"
)
print(f"n_directional     = {vR['n_directional']}   frac = {vR['frac_directional']:.4f}")
print(f"dominant_direction= {vR['dominant_direction']}   panel_agrees = {vR['panel_agrees']}")
print(
    f"panel TY p: quality->volume = {vR['panel_p_quality_leads']:.4g}  "
    f"volume->quality = {vR['panel_p_quality_lags']:.4g}"
)
print(f"\nLEVELS TEST HOLDS (rct_share) = {vR['holds']}")

LEVELS_AGREE = bool(vP["holds"] == vR["holds"])
print(
    f"\nPrimary and RCT-share levels verdicts AGREE = {LEVELS_AGREE}  "
    f"(primary holds={vP['holds']}, rct_share holds={vR['holds']})"
)

=== LEVELS ROBUSTNESS (rct_share, Toda-Yamamoto) ===
n_qualifying      = 138
n_non_evaluable   = 0   (expected 0; gap-free series)
n_evaluable       = 138   (pooled p-values = 276)
direction split   : lead=0  lag=0  coupled=0  none=138
n_directional     = 0   frac = 0.0000
dominant_direction= none   panel_agrees = False
panel TY p: quality->volume = 0.8527  volume->quality = 0.6518

LEVELS TEST HOLDS (rct_share) = False

Primary and RCT-share levels verdicts AGREE = True  (primary holds=False, rct_share holds=False)


## 4. Differenced CONTROL — reproduce the PR3 / F1 differenced null on the SAME 138 topics

Re-run the existing `granger_pair` / `panel_granger` **differenced** pipeline (notebook 11's
`run_f1`, cloned verbatim) on the SAME universe. This proves the harness here is wired identically
to notebook 11 and gives a clean differenced-vs-levels side-by-side. We cross-check the reproduced
numbers against the locked `data/v1/f1_cascade_verdict.json` (0/138 directional, panel
p≈0.277/0.739 for mean_tier).

In [7]:
def run_f1_differenced(frame: pd.DataFrame, *, quality_col: str) -> dict:
    """Notebook-11 differenced F1 loop, cloned verbatim (the PR3 control)."""
    series = C.build_topic_year_series(frame)
    topics = C.qualifying_topics(series)
    n_qualifying = len(topics)

    pooled_pvals: list[float] = []
    differenced: list[tuple] = []
    per_topic: list[dict] = []

    for topic in topics:
        topic_series = series[series.topic_id == topic]
        quality, volume, status = C.prepare_series(topic_series, quality_col=quality_col)
        if status == "ok":
            p_qv, p_vq = C.granger_pair(quality, volume)  # differences internally
            pos_qv = len(pooled_pvals)
            pooled_pvals.append(p_qv)
            pos_vq = len(pooled_pvals)
            pooled_pvals.append(p_vq)
            differenced.append((np.diff(quality.to_numpy()), np.diff(volume.to_numpy())))
            per_topic.append(
                {"topic_id": int(topic), "direction": "none", "_pos_qv": pos_qv, "_pos_vq": pos_vq}
            )
        else:
            per_topic.append(
                {"topic_id": int(topic), "direction": "none", "_pos_qv": None, "_pos_vq": None}
            )

    reject = C.bh_fdr(np.array(pooled_pvals, dtype="float64"))
    for row in per_topic:
        if row["_pos_qv"] is not None:
            row["direction"] = C.classify_direction(
                leads_sig=bool(reject[row["_pos_qv"]]), lags_sig=bool(reject[row["_pos_vq"]])
            )
    panel = C.panel_granger(differenced)
    verdict = C.decide_f1(per_topic, panel, n_qualifying=n_qualifying)
    return {"verdict": verdict, "panel": (float(panel[0]), float(panel[1]))}


diff_primary = run_f1_differenced(df, quality_col="mean_tier")
diff_rct = run_f1_differenced(df, quality_col="rct_share")
dvP, dvR = diff_primary["verdict"], diff_rct["verdict"]

print("=== DIFFERENCED CONTROL (reproducing PR3 / F1 null) ===")
print(
    f"mean_tier : holds={dvP['holds']}  n_directional={dvP['n_directional']}/138  "
    f"panel q->v={dvP['panel_p_quality_leads']:.4g}  v->q={dvP['panel_p_quality_lags']:.4g}"
)
print(
    f"rct_share : holds={dvR['holds']}  n_directional={dvR['n_directional']}/138  "
    f"panel q->v={dvR['panel_p_quality_leads']:.4g}  v->q={dvR['panel_p_quality_lags']:.4g}"
)

=== DIFFERENCED CONTROL (reproducing PR3 / F1 null) ===
mean_tier : holds=False  n_directional=0/138  panel q->v=0.2771  v->q=0.7393
rct_share : holds=False  n_directional=0/138  panel q->v=0.9502  v->q=0.9562


In [8]:
# Cross-check the reproduced differenced control against the LOCKED f1_cascade_verdict.json.
locked = json.loads((DATA / "f1_cascade_verdict.json").read_text())
lk = locked["primary"]
lk_panel = locked["panel_primary"]
match_dir = dvP["n_directional"] == lk["n_directional"]
match_holds = dvP["holds"] == lk["holds"]
match_panel = np.isclose(
    dvP["panel_p_quality_leads"], lk_panel["p_quality_leads"], atol=1e-6
) and np.isclose(dvP["panel_p_quality_lags"], lk_panel["p_quality_lags"], atol=1e-6)
DIFF_REPRO_OK = bool(match_dir and match_holds and match_panel)
rep_dir = dvP["n_directional"]
rep_qlead = dvP["panel_p_quality_leads"]
rep_qlag = dvP["panel_p_quality_lags"]
lk_dir = lk["n_directional"]
lk_qlead = lk_panel["p_quality_leads"]
lk_qlag = lk_panel["p_quality_lags"]
print("Differenced-control cross-check vs locked f1_cascade_verdict.json:")
print(f"  n_directional reproduced {rep_dir} == locked {lk_dir}: {match_dir}")
print(f"  holds reproduced {dvP['holds']} == locked {lk['holds']}: {match_holds}")
print(
    f"  panel reproduced ({rep_qlead:.6f}, {rep_qlag:.6f}) "
    f"== locked ({lk_qlead:.6f}, {lk_qlag:.6f}): {match_panel}"
)
assert DIFF_REPRO_OK, "Differenced control does NOT reproduce the locked F1 null — mismatch!"
print("\nDIFFERENCED CONTROL REPRODUCES THE LOCKED F1 NULL — harness wired as in notebook 11.")

Differenced-control cross-check vs locked f1_cascade_verdict.json:
  n_directional reproduced 0 == locked 0: True
  holds reproduced False == locked False: True
  panel reproduced (0.277125, 0.739280) == locked (0.277125, 0.739280): True

DIFFERENCED CONTROL REPRODUCES THE LOCKED F1 NULL — harness wired as in notebook 11.


## 5. Engle-Granger ECM robustness summary (informational; cannot flip the verdict)

Per evaluable topic (mean_tier primary) we summarize how many topics cointegrate
(`coint_pvalue < 0.05`) and the error-correction (speed-of-adjustment) coefficient sign / p
distribution among cointegrated topics. A significant **negative** `ec_coef` means volume adjusts
toward a long-run equilibrium with quality. PR3D §3.3 / §9: this is ROBUSTNESS / INFORMATIONAL
ONLY — it characterizes whether a long-run level relationship exists but CANNOT flip HOLD/NULL.

In [9]:
ecm_rows = rdfP[rdfP["status"] == "ok"].copy()
n_ecm_eval = len(ecm_rows)
coint_p = ecm_rows["coint_pvalue"].to_numpy()
n_coint_p_ok = int(np.isfinite(coint_p).sum())
coint_mask = ecm_rows["cointegrated"].to_numpy()
n_coint = int(coint_mask.sum())

coint_rows = ecm_rows[ecm_rows["cointegrated"]]
ec_coef = coint_rows["ec_coef"].to_numpy()
ec_p = coint_rows["ec_pvalue"].to_numpy()
fin = np.isfinite(ec_coef) & np.isfinite(ec_p)
n_ec_fin = int(fin.sum())
n_ec_neg = int((ec_coef[fin] < 0).sum())
n_ec_neg_sig = int(((ec_coef[fin] < 0) & (ec_p[fin] < 0.05)).sum())
n_ec_pos_sig = int(((ec_coef[fin] >= 0) & (ec_p[fin] < 0.05)).sum())
pct_coint = 100 * n_coint / max(n_ecm_eval, 1)

print("=== Engle-Granger ECM summary (mean_tier primary; INFORMATIONAL ONLY) ===")
print(f"evaluable topics with a coint p-value : {n_coint_p_ok}/{n_ecm_eval}")
print(f"cointegrated (coint_pvalue<0.05)       : {n_coint}/{n_ecm_eval}  ({pct_coint:.1f}%)")
print(f"  of cointegrated, EC coef computed    : {n_ec_fin}")
print(f"    negative EC coef (adjusts toward eq): {n_ec_neg}/{n_ec_fin}")
print(f"    negative AND significant (p<0.05)   : {n_ec_neg_sig}")
print(f"    positive AND significant (p<0.05)   : {n_ec_pos_sig}")
if n_ec_fin:
    print(
        f"  EC coef distribution (cointegrated): "
        f"median={np.median(ec_coef[fin]):.4f}  "
        f"min={ec_coef[fin].min():.4f}  max={ec_coef[fin].max():.4f}"
    )
print(
    "\nINFORMATIONAL ONLY (PR3D §3.3/§9): a favorable ECM result cannot convert a NULL "
    "levels verdict into a HOLD."
)

=== Engle-Granger ECM summary (mean_tier primary; INFORMATIONAL ONLY) ===
evaluable topics with a coint p-value : 126/128
cointegrated (coint_pvalue<0.05)       : 50/128  (39.1%)
  of cointegrated, EC coef computed    : 50
    negative EC coef (adjusts toward eq): 49/50
    negative AND significant (p<0.05)   : 27
    positive AND significant (p<0.05)   : 0
  EC coef distribution (cointegrated): median=-0.4964  min=-1.3454  max=0.0081

INFORMATIONAL ONLY (PR3D §3.3/§9): a favorable ECM result cannot convert a NULL levels verdict into a HOLD.


## 6. Power simulation (DECISIVE; PR3D §7, locked grid, run ONCE)

`run_powersim()` with the locked defaults: 4 DGPs × 5 T × 5 β × 2 tests at `n_rep=500`,
`lag=2`, `alpha=0.05`, `seed=20260609`. For every replication BOTH the PR3 differenced
`granger_pair` and the TY levels `toda_yamamoto_pair` run on the SAME series. This establishes,
on synthetic ground truth, which test detects which cascade at the real series lengths, and
measures each test's β=0 size per T so any TY short-T over-rejection is **measured, not hidden**.
Takes ~10-20 min; run once, n_rep is NOT reduced below the locked 500.

In [10]:
import time as _time

_t0 = _time.time()
powersim = PS.run_powersim(
    T_grid=tuple(POWERSIM_GRID["T_grid"]),
    beta_grid=tuple(POWERSIM_GRID["beta_grid"]),
    dgps=tuple(POWERSIM_GRID["dgps"]),
    n_rep=POWERSIM_GRID["n_rep"],
    lag=POWERSIM_GRID["lag"],
    alpha=POWERSIM_GRID["alpha"],
    seed=POWERSIM_GRID["seed"],
)
print(f"power simulation done in {_time.time() - _t0:.1f}s  | rows = {len(powersim)}")
assert (powersim["n_rep"] == 500).all(), "n_rep must be the locked 500 in every cell"
print(powersim.groupby(["dgp", "test"], as_index=False).size().to_string(index=False))

power simulation done in 53.9s  | rows = 200
                dgp        test  size
       diff_cascade differenced    25
       diff_cascade      levels    25
      level_cascade differenced    25
      level_cascade      levels    25
level_cascade_coint differenced    25
level_cascade_coint      levels    25
               null differenced    25
               null      levels    25


In [11]:
# Honesty check (b) + the size table: the beta=0 (null/size) row per T for BOTH tests, and the
# headline level_cascade_coint @ T=30 power read the verdict routes through (PR3D §8.1, §7.3).


def _cell(dgp: str, t_len: int, beta: float, test: str) -> float:
    """Look up one detection-rate cell from the power-sim grid."""
    sel = powersim[
        (powersim["dgp"] == dgp)
        & (powersim["T"] == t_len)
        & np.isclose(powersim["beta"], beta)
        & (powersim["test"] == test)
    ]
    return float(sel["detect_rate"].iloc[0]) if len(sel) else float("nan")


print("=== SIZE (beta=0 false-positive rate) per T — measured, NOT hidden (PR3D §7.3) ===")
print(f"{'T':>4} | {'differenced':>12} | {'TY levels':>12}   (null DGP, beta=0)")
for t_len in POWERSIM_GRID["T_grid"]:
    sd = _cell("null", t_len, 0.0, "differenced")
    sl = _cell("null", t_len, 0.0, "levels")
    flag = ""
    if t_len < 30 and sl > 0.15:
        flag = "  <- TY over-rejects (excluded from power read)"
    if t_len == 30:
        flag = "  <- headline T (size-controlling)"
    print(f"{t_len:>4} | {sd:>12.3f} | {sl:>12.3f}{flag}")

print("\n=== HEADLINE power read: level_cascade_coint @ T=30 (the ONLY verdict cell) ===")
print(f"{'beta':>5} | {'differenced':>12} | {'TY levels':>12}")
coint_T30 = {}
for beta in POWERSIM_GRID["beta_grid"]:
    dd = _cell("level_cascade_coint", 30, beta, "differenced")
    dl = _cell("level_cascade_coint", 30, beta, "levels")
    coint_T30[beta] = {"differenced": dd, "levels": dl}
    print(f"{beta:>5.2f} | {dd:>12.3f} | {dl:>12.3f}")

=== SIZE (beta=0 false-positive rate) per T — measured, NOT hidden (PR3D §7.3) ===
   T |  differenced |    TY levels   (null DGP, beta=0)
   8 |        0.000 |        0.000
  15 |        0.050 |        0.336  <- TY over-rejects (excluded from power read)
  22 |        0.044 |        0.144
  30 |        0.052 |        0.104  <- headline T (size-controlling)
  40 |        0.050 |        0.074

=== HEADLINE power read: level_cascade_coint @ T=30 (the ONLY verdict cell) ===
 beta |  differenced |    TY levels
 0.00 |        0.068 |        0.092
 0.25 |        0.050 |        0.090
 0.50 |        0.076 |        0.158
 1.00 |        0.144 |        0.458
 2.00 |        0.382 |        0.884


## 7. Apply the PR3D §8.1 verdict table MECHANICALLY (once)

The power read is taken off the **`level_cascade_coint`** DGP at the **`T=30`** cell ONLY (the only
length where the TY χ² Wald controls size; T=8/15/22 reported but EXCLUDED, PR3D §7.3). "Levels
well-powered" = the TY detection rate at T=30 reaches a high level at the larger β where the
differenced test stays at/near size (the qualitative contrast PR3D HANDOFF FLAG 5 specifies — no
post-hoc numeric cutoff is invented). We surface the per-β detection rates so the verdict is
auditable, then apply the symmetric table:

- **APPARENT** iff levels HOLDS (mean_tier) AND robust (rct_share) AND differenced under-powered
  while levels well-powered for `level_cascade_coint` @ T=30.
- **TRUE** iff levels NULL (mean_tier) AND levels well-powered for `level_cascade_coint` @ T=30.
- **AMBIGUOUS** otherwise.

In [12]:
# --- Power read at the single locked cell (level_cascade_coint @ T=30). ---
# "differenced under-powered" = differenced detection stays near size across beta (BLIND to the
# cointegrated/integrator level cascade). "levels well-powered" = TY detection rises to a clearly
# high level at the larger beta. Both read qualitatively (PR3D HANDOFF FLAG 5) from coint_T30.
diff_rates = np.array([coint_T30[b]["differenced"] for b in POWERSIM_GRID["beta_grid"]])
lev_rates = np.array([coint_T30[b]["levels"] for b in POWERSIM_GRID["beta_grid"]])
diff_max = float(diff_rates.max())  # best differenced detection across beta at the coint cell
lev_max = float(lev_rates.max())  # best TY detection across beta at the coint cell
ty_size_T30 = _cell("null", 30, 0.0, "levels")  # TY size at the headline length

# Qualitative contrast (auditable, not a retrofitted numeric bar): the differenced test is
# under-powered (its best coint-cell detection stays low / near its own size) WHILE the TY levels
# test reaches a clearly high detection rate at large beta well above its T=30 size.
DIFFERENCED_UNDERPOWERED_COINT_T30 = bool(diff_max < 0.30)
LEVELS_WELLPOWERED_COINT_T30 = bool(lev_max >= 0.80 and lev_max > ty_size_T30 + 0.30)

diff_list = np.round(diff_rates, 3).tolist()
lev_list = np.round(lev_rates, 3).tolist()
print("Power read @ level_cascade_coint, T=30 (the ONLY verdict cell, PR3D §8.1):")
print(f"  differenced detection across beta : {diff_list}  (max {diff_max:.3f})")
print(f"  TY levels  detection across beta  : {lev_list}  (max {lev_max:.3f})")
print(f"  TY size at T=30 (null, beta=0)    : {ty_size_T30:.3f}")
print(f"  -> differenced UNDER-powered @coint/T30 = {DIFFERENCED_UNDERPOWERED_COINT_T30}")
print(f"  -> TY levels WELL-powered @coint/T30    = {LEVELS_WELLPOWERED_COINT_T30}")

# Effect-size nuance for honesty check (b): at which beta does TY first clear 0.80?
betas_arr = np.array(POWERSIM_GRID["beta_grid"])
clears = betas_arr[lev_rates >= 0.80]
ty_min_beta_powered = float(clears.min()) if clears.size else float("nan")
print(
    f"  TY first reaches >=0.80 detection at beta = {ty_min_beta_powered}  "
    "(below this, only LARGE cointegrated cascades are reliably detected)"
)

Power read @ level_cascade_coint, T=30 (the ONLY verdict cell, PR3D §8.1):
  differenced detection across beta : [0.068, 0.05, 0.076, 0.144, 0.382]  (max 0.382)
  TY levels  detection across beta  : [0.092, 0.09, 0.158, 0.458, 0.884]  (max 0.884)
  TY size at T=30 (null, beta=0)    : 0.104
  -> differenced UNDER-powered @coint/T30 = False
  -> TY levels WELL-powered @coint/T30    = True
  TY first reaches >=0.80 detection at beta = 2.0  (below this, only LARGE cointegrated cascades are reliably detected)


In [13]:
# --- Mechanical application of the PR3D §8.1 verdict table (symmetric; applied ONCE). ---
levels_holds_primary = bool(vP["holds"])
levels_holds_rct = bool(vR["holds"])
levels_robust = bool(levels_holds_primary and levels_holds_rct)

apparent = bool(
    levels_holds_primary
    and levels_robust
    and DIFFERENCED_UNDERPOWERED_COINT_T30
    and LEVELS_WELLPOWERED_COINT_T30
)
true_null = bool((not levels_holds_primary) and LEVELS_WELLPOWERED_COINT_T30)

if apparent:
    VERDICT = "APPARENT"
elif true_null:
    VERDICT = "TRUE"
else:
    VERDICT = "AMBIGUOUS"

print("=" * 78)
print(f"PR3D §8.1 VERDICT (applied once, mechanically): F1 levels null is **{VERDICT}**")
print("=" * 78)
print(f"  levels HOLDS (mean_tier)            = {levels_holds_primary}")
print(f"  levels robust (rct_share also holds)= {levels_robust}")
print(f"  differenced under-powered @coint/T30= {DIFFERENCED_UNDERPOWERED_COINT_T30}")
print(f"  levels well-powered @coint/T30      = {LEVELS_WELLPOWERED_COINT_T30}")
print(f"  short-T size-inflation flag (a)     = {SHORT_T_FLAG}")
if VERDICT == "TRUE":
    print(
        "\n  => F1's null is a TRUE level-null: the levels test is NULL on mean_tier AND the\n"
        "     power-sim shows the TY levels test WOULD have detected a (large) cointegrated\n"
        "     level-cascade at T=30. This STRENGTHENS PR3's signed null."
    )
elif VERDICT == "APPARENT":
    print(
        "\n  => F1's null is APPARENT: the differenced spec was blind to a level-cascade the\n"
        "     (well-powered, robust) levels test detects. Diagnostic SIGNAL only — a flip needs\n"
        "     a separate PR4 + additive G5' + Samer's Phase-2 checkpoint (PR3D §8.2)."
    )
else:
    print(
        "\n  => AMBIGUOUS: the levels diagnostic could not decisively distinguish a true level-\n"
        "     null from an apparent one under the pre-stated conjunction (a fully reportable\n"
        "     outcome, PR3D §8.1)."
    )
print("\n  does_not_flip_g5 = True  (PR3D §8.2 — this diagnostic CANNOT flip Gate G5)")

PR3D §8.1 VERDICT (applied once, mechanically): F1 levels null is **TRUE**
  levels HOLDS (mean_tier)            = False
  levels robust (rct_share also holds)= False
  differenced under-powered @coint/T30= False
  levels well-powered @coint/T30      = True
  short-T size-inflation flag (a)     = True

  => F1's null is a TRUE level-null: the levels test is NULL on mean_tier AND the
     power-sim shows the TY levels test WOULD have detected a (large) cointegrated
     level-cascade at T=30. This STRENGTHENS PR3's signed null.

  does_not_flip_g5 = True  (PR3D §8.2 — this diagnostic CANNOT flip Gate G5)


## 8. Figure — `docs/figures/F1_levels_powersim.png`

Power curves (detection rate vs β per T, both tests, all four DGPs). The `level_cascade_coint`
@ T=30 cell the verdict reads off is clearly marked; the β=0 size column per T is shown so the
TY short-T over-rejection is **visible, not hidden** (PR3D §7.3).

In [14]:
DGP_ORDER = ["null", "level_cascade", "diff_cascade", "level_cascade_coint"]
DGP_TITLE = {
    "null": "(a) null — size / false-positive",
    "level_cascade": "(b) level_cascade — both detect",
    "diff_cascade": "(c) diff_cascade — fairness (differenced)",
    "level_cascade_coint": "(d) level_cascade_coint — VERDICT DGP",
}
T_COLORS = {8: "#9aa0a6", 15: "#fbbc04", 22: "#34a853", 30: "#ea4335", 40: "#4285f4"}
TEST_STYLE = {"differenced": dict(ls="--", marker="s"), "levels": dict(ls="-", marker="o")}

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5))
for ax, dgp in zip(axes.ravel(), DGP_ORDER, strict=True):
    for t_len in POWERSIM_GRID["T_grid"]:
        for test, style in TEST_STYLE.items():
            rates = [_cell(dgp, t_len, b, test) for b in POWERSIM_GRID["beta_grid"]]
            ax.plot(
                POWERSIM_GRID["beta_grid"],
                rates,
                color=T_COLORS[t_len],
                lw=1.5 if test == "levels" else 1.1,
                alpha=0.95 if test == "levels" else 0.7,
                **style,
            )
    ax.axhline(0.05, ls=":", c="#444444", lw=1)  # nominal alpha
    ax.set_xlabel("effect size β")
    ax.set_ylabel("detection rate")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(DGP_TITLE[dgp], fontsize=9.5)
    # Mark the verdict cell: level_cascade_coint @ T=30 (red, levels line).
    if dgp == "level_cascade_coint":
        rr = [_cell(dgp, 30, b, "levels") for b in POWERSIM_GRID["beta_grid"]]
        ax.scatter(
            POWERSIM_GRID["beta_grid"],
            rr,
            s=120,
            facecolors="none",
            edgecolors="#ea4335",
            linewidths=1.8,
            zorder=5,
        )
        ax.annotate(
            "verdict read:\nTY levels (solid red)\nvs differenced (dashed red)\n@ T=30",
            xy=(0.97, 0.03),
            xycoords="axes fraction",
            ha="right",
            va="bottom",
            fontsize=7.5,
            bbox=dict(boxstyle="round", fc="#fff3cd", ec="#ea4335", alpha=0.9),
        )

# Shared legend: T colors + test linestyles.
t_handles = [
    Line2D([0], [0], color=T_COLORS[t], lw=2, label=f"T={t}") for t in POWERSIM_GRID["T_grid"]
]
style_handles = [
    Line2D([0], [0], color="#444444", ls="-", marker="o", label="TY levels test"),
    Line2D([0], [0], color="#444444", ls="--", marker="s", label="differenced (PR3) test"),
    Line2D([0], [0], color="#444444", ls=":", label="nominal α = 0.05"),
]
fig.legend(
    handles=t_handles + style_handles,
    ncol=8,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.01),
    fontsize=8,
    frameon=False,
)
suptitle = (
    "F1 levels diagnostic — power simulation (differenced vs Toda-Yamamoto levels)  |  "
    f"verdict = {VERDICT}  (read off level_cascade_coint @ T=30; "
    f"TY size @ T=30 = {ty_size_T30:.2f})"
)
fig.suptitle(suptitle, fontsize=11.5)
fig.tight_layout(rect=[0, 0.04, 1, 0.97])
POWERSIM_FIG = FIGURES_DIR / "F1_levels_powersim.png"
fig.savefig(POWERSIM_FIG, dpi=DPI, bbox_inches="tight")
plt.close(fig)
sz = POWERSIM_FIG.stat().st_size
print(f"wrote {POWERSIM_FIG} | {sz / 1024:.1f} KB (dpi={DPI})")
assert sz < 1_000_000, f"figure too large: {sz} bytes"
print("figure size assertion PASS — under 1 MB")

wrote /Users/samersalman/Desktop/SciField/docs/figures/F1_levels_powersim.png | 269.7 KB (dpi=120)
figure size assertion PASS — under 1 MB


## 9. Persist artifacts (+ provenance sidecars)

Writes the three declared PR3D outputs (PR3D §10): `f1_levels_diagnostic_results.parquet`
(per-topic levels results, both quality metrics), `f1_levels_diagnostic_verdict.json` (HOLD/NULL
verdicts + level-panel p-values + the §8 TRUE/APPARENT/AMBIGUOUS verdict + `does_not_flip_g5`),
and `f1_levels_powersim.parquet` (the full detection-rate grid incl. β=0 size rows), each with a
`record_run` sidecar. The figure sidecar is written too.

In [15]:
# (a) Per-topic levels results — BOTH quality metrics, tagged by `quality_col`.
RESULT_COLS = [
    "quality_col",
    "topic_id",
    "status",
    "series_len",
    "p_q_to_v",
    "p_v_to_q",
    "leads_sig",
    "lags_sig",
    "direction",
    "coint_pvalue",
    "cointegrated",
    "ec_coef",
    "ec_pvalue",
]
res_primary = rdfP.copy()
res_primary.insert(0, "quality_col", "mean_tier")
res_rct = rct["results_df"].copy()
res_rct.insert(0, "quality_col", "rct_share")
results_all = pd.concat([res_primary, res_rct], ignore_index=True)[RESULT_COLS]
results_all["topic_id"] = results_all["topic_id"].astype("int64")
results_all["series_len"] = results_all["series_len"].astype("int64")

RESULTS_PARQUET = DATA / "f1_levels_diagnostic_results.parquet"
results_all.to_parquet(RESULTS_PARQUET, index=False)
print("wrote", RESULTS_PARQUET, "| rows =", len(results_all))

results_sidecar = record_run(
    artifact_path=RESULTS_PARQUET,
    inputs={
        "epistemic": DATA / "epistemic_extracted.parquet",
        "novelty": DATA / "novelty_semantic.parquet",
    },
    config={
        "artifact": "f1_levels_diagnostic_results",
        "session": "V1-S15 (PR3D diagnostic)",
        "prereg": "PR3D_levels_diagnostic",
        "quality_series": ["mean_tier (PRIMARY)", "rct_share (robustness)"],
        "n_rows": int(len(results_all)),
        "n_qualifying": int(primary["n_qualifying"]),
        "n_evaluable_primary": int(primary["n_evaluable"]),
        "n_non_evaluable_primary": int(primary["n_non_evaluable"]),
        "params": PR3D_PARAMS,
    },
)
print("recorded results sidecar:", results_sidecar)
results_all.head()

wrote /Users/samersalman/Desktop/SciField/data/v1/f1_levels_diagnostic_results.parquet | rows = 276
recorded results sidecar: /Users/samersalman/Desktop/SciField/data/v1/f1_levels_diagnostic_results.parquet.run.json


,quality_col,topic_id,status,series_len,p_q_to_v,p_v_to_q,leads_sig,lags_sig,direction,coint_pvalue,cointegrated,ec_coef,ec_pvalue
0,mean_tier,0,ok,32,0.052327,0.051683,False,False,none,3.883276e-01,False,NaN,NaN
1,mean_tier,1,ok,32,0.154902,0.073353,False,False,none,2.016891e-07,True,-1.043269,0.003471
2,mean_tier,2,ok,31,0.136167,0.865772,False,False,none,8.569557e-01,False,NaN,NaN
3,mean_tier,3,ok,32,0.932837,0.003895,False,False,none,9.144129e-02,False,NaN,NaN
4,mean_tier,4,ok,31,0.979987,0.168042,False,False,none,9.050470e-01,False,NaN,NaN


In [16]:
# (b) Power-sim detection-rate grid (the full 4-DGP x 5-T x 5-beta x 2-test grid).
POWERSIM_PARQUET = DATA / "f1_levels_powersim.parquet"
powersim.to_parquet(POWERSIM_PARQUET, index=False)
print("wrote", POWERSIM_PARQUET, "| rows =", len(powersim))

ty_size_null = {str(t): float(_cell("null", t, 0.0, "levels")) for t in POWERSIM_GRID["T_grid"]}
diff_size_null = {
    str(t): float(_cell("null", t, 0.0, "differenced")) for t in POWERSIM_GRID["T_grid"]
}
powersim_sidecar = record_run(
    artifact_path=POWERSIM_PARQUET,
    inputs={},
    config={
        "artifact": "f1_levels_powersim",
        "session": "V1-S15 (PR3D diagnostic)",
        "prereg": "PR3D_levels_diagnostic",
        "grid": POWERSIM_GRID,
        "n_cells": int(len(powersim)),
        "ty_size_by_T_null": ty_size_null,
        "differenced_size_by_T_null": diff_size_null,
    },
)
print("recorded powersim sidecar:", powersim_sidecar)

wrote /Users/samersalman/Desktop/SciField/data/v1/f1_levels_powersim.parquet | rows = 200
recorded powersim sidecar: /Users/samersalman/Desktop/SciField/data/v1/f1_levels_powersim.parquet.run.json


In [17]:
# (c) Machine-readable verdict JSON — the §8 verdict, both HOLD/NULL verdicts, panels, power read.
VERDICT_JSON = DATA / "f1_levels_diagnostic_verdict.json"
diff_by_beta = {str(b): coint_T30[b]["differenced"] for b in POWERSIM_GRID["beta_grid"]}
lev_by_beta = {str(b): coint_T30[b]["levels"] for b in POWERSIM_GRID["beta_grid"]}
dir_T_median = int(np.median(dir_rows["series_len"])) if len(dir_rows) else None
verdict_payload = {
    "diagnostic": "PR3D_levels_diagnostic",
    "verdict": VERDICT,
    "does_not_flip_g5": True,
    "levels_primary_mean_tier": vP,
    "levels_rct_share": vR,
    "levels_verdicts_agree": LEVELS_AGREE,
    "levels_panel_primary": {
        "p_quality_leads": primary["panel"][0],
        "p_quality_lags": primary["panel"][1],
    },
    "levels_panel_rct_share": {
        "p_quality_leads": rct["panel"][0],
        "p_quality_lags": rct["panel"][1],
    },
    "differenced_control": {
        "mean_tier": dvP,
        "rct_share": dvR,
        "reproduces_locked_f1_null": DIFF_REPRO_OK,
    },
    "ecm_summary_mean_tier": {
        "n_evaluable": int(n_ecm_eval),
        "n_cointegrated": int(n_coint),
        "frac_cointegrated": float(n_coint / max(n_ecm_eval, 1)),
        "n_ec_neg": int(n_ec_neg),
        "n_ec_neg_sig": int(n_ec_neg_sig),
        "n_ec_pos_sig": int(n_ec_pos_sig),
    },
    "powersim_verdict_read": {
        "dgp": "level_cascade_coint",
        "T": 30,
        "differenced_detect_by_beta": diff_by_beta,
        "levels_detect_by_beta": lev_by_beta,
        "ty_size_T30": float(ty_size_T30),
        "differenced_underpowered": DIFFERENCED_UNDERPOWERED_COINT_T30,
        "levels_wellpowered": LEVELS_WELLPOWERED_COINT_T30,
        "ty_min_beta_powered": ty_min_beta_powered,
    },
    "honesty_checks": {
        "short_T_size_inflation_flag": SHORT_T_FLAG,
        "n_directional_primary": int(vP["n_directional"]),
        "directional_topic_T_median": dir_T_median,
        "evaluable_T_median": int(T_eval_median),
        "ty_size_caveat": (
            "TY power for level_cascade_coint @ T=30 is effect-size dependent; well-powered "
            "only for LARGE cascades, so a levels NULL rules out a large level-cascade, not "
            "necessarily a small one."
        ),
    },
    "n_universe_papers": int(n_leaf),
    "n_leaf_topics": int(n_leaf_topics),
    "n_qualifying": int(primary["n_qualifying"]),
    "params": PR3D_PARAMS,
}
VERDICT_JSON.write_text(json.dumps(verdict_payload, indent=2, sort_keys=False))
print("wrote", VERDICT_JSON)

verdict_sidecar = record_run(
    artifact_path=VERDICT_JSON,
    inputs={
        "epistemic": DATA / "epistemic_extracted.parquet",
        "novelty": DATA / "novelty_semantic.parquet",
    },
    config={
        "artifact": "f1_levels_diagnostic_verdict",
        "session": "V1-S15 (PR3D diagnostic)",
        "prereg": "PR3D_levels_diagnostic",
        "verdict": VERDICT,
        "does_not_flip_g5": True,
        "levels_holds_primary": levels_holds_primary,
        "levels_holds_rct_share": levels_holds_rct,
        "params": PR3D_PARAMS,
    },
)
print("recorded verdict sidecar:", verdict_sidecar)

wrote /Users/samersalman/Desktop/SciField/data/v1/f1_levels_diagnostic_verdict.json
recorded verdict sidecar: /Users/samersalman/Desktop/SciField/data/v1/f1_levels_diagnostic_verdict.json.run.json


In [18]:
# (d) Figure provenance sidecar.
fig_sidecar = record_run(
    artifact_path=POWERSIM_FIG,
    inputs={},
    config={
        "figure": "F1_levels_powersim",
        "session": "V1-S15 (PR3D diagnostic)",
        "prereg": "PR3D_levels_diagnostic",
        "dpi": DPI,
        "grid": POWERSIM_GRID,
        "verdict": VERDICT,
        "verdict_read_cell": "level_cascade_coint @ T=30",
    },
)
print("recorded figure sidecar:", fig_sidecar)

recorded figure sidecar: /Users/samersalman/Desktop/SciField/docs/figures/F1_levels_powersim.png.run.json


## 10. Verify artifacts + confirm locked `f1_cascade_*` byte-unchanged

In [19]:
import hashlib

artifacts = [
    RESULTS_PARQUET,
    RESULTS_PARQUET.with_suffix(RESULTS_PARQUET.suffix + ".run.json"),
    POWERSIM_PARQUET,
    POWERSIM_PARQUET.with_suffix(POWERSIM_PARQUET.suffix + ".run.json"),
    VERDICT_JSON,
    VERDICT_JSON.with_suffix(VERDICT_JSON.suffix + ".run.json"),
    POWERSIM_FIG,
    POWERSIM_FIG.with_suffix(POWERSIM_FIG.suffix + ".run.json"),
]
for a in artifacts:
    ok = a.exists()
    kb = f"{a.stat().st_size / 1024:.1f} KB" if ok else "MISSING"
    print(f"  [{'ok' if ok else 'XX'}] {a.name:<44} {kb}")
    assert ok, f"artifact missing: {a}"
assert POWERSIM_FIG.stat().st_size < 1_000_000, "figure exceeds 1 MB"

# Confirm the LOCKED f1_cascade artifacts (signed-gate inputs) are byte-unchanged. These are the
# baseline sha256s captured before this diagnostic ran; PR3D is additive (PR3D §11).
LOCKED_SHA = {
    "f1_cascade_results.parquet": (
        "c870ccce0c5edf519114bf3d6e5528fc6441562c2c99c9b0d458cfd0a4cbcefb"
    ),
    "f1_cascade_verdict.json": ("7cf312f30b6d3cefddc4c1fadeba40651b1d6dfb9f95378939d31117a330a747"),
}
print("\nLocked f1_cascade_* byte-unchanged check (PR3D is additive):")
all_unchanged = True
for name, want in LOCKED_SHA.items():
    got = hashlib.sha256((DATA / name).read_bytes()).hexdigest()
    same = got == want
    all_unchanged = all_unchanged and same
    print(f"  [{'ok' if same else 'XX'}] {name:<32} {'UNCHANGED' if same else 'CHANGED!'}")
assert all_unchanged, "LOCKED f1_cascade artifact changed — PR3D must be additive only!"
print("\nAll PR3D artifacts present (figure < 1 MB); locked f1_cascade_* byte-unchanged.")

  [ok] f1_levels_diagnostic_results.parquet         17.6 KB
  [ok] f1_levels_diagnostic_results.parquet.run.json 1.4 KB
  [ok] f1_levels_powersim.parquet                   4.7 KB
  [ok] f1_levels_powersim.parquet.run.json          1.2 KB
  [ok] f1_levels_diagnostic_verdict.json            3.5 KB
  [ok] f1_levels_diagnostic_verdict.json.run.json   1.3 KB
  [ok] F1_levels_powersim.png                       269.7 KB
  [ok] F1_levels_powersim.png.run.json              1.1 KB

Locked f1_cascade_* byte-unchanged check (PR3D is additive):
  [ok] f1_cascade_results.parquet       UNCHANGED
  [ok] f1_cascade_verdict.json          UNCHANGED

All PR3D artifacts present (figure < 1 MB); locked f1_cascade_* byte-unchanged.


## 11. Final summary (for the Phase-2 human checkpoint)

In [20]:
ty_size_line = "  ".join(
    f"T{t}={_cell('null', t, 0.0, 'levels'):.2f}" for t in POWERSIM_GRID["T_grid"]
)
dir_T_med_str = str(int(np.median(dir_rows["series_len"]))) if len(dir_rows) else "n/a"
print("=" * 78)
print(f"F1 LEVELS DIAGNOSTIC (PR3D) — verdict: {VERDICT}  | does_not_flip_g5 = True")
print("=" * 78)
print(
    f"LEVELS mean_tier : HOLDS={vP['holds']}  directional={vP['n_directional']}/138 "
    f"(dominant {vP['dominant_direction']})  panel q->v={vP['panel_p_quality_leads']:.4g} "
    f"v->q={vP['panel_p_quality_lags']:.4g}"
)
print(
    f"LEVELS rct_share : HOLDS={vR['holds']}  directional={vR['n_directional']}/138  "
    f"panel q->v={vR['panel_p_quality_leads']:.4g} v->q={vR['panel_p_quality_lags']:.4g}"
)
print(
    f"DIFFERENCED ctrl : mean_tier {dvP['n_directional']}/138 "
    f"(panel {dvP['panel_p_quality_leads']:.4g}/{dvP['panel_p_quality_lags']:.4g}) — "
    f"reproduces locked F1 null = {DIFF_REPRO_OK}"
)
print(
    f"ECM (mean_tier)  : {n_coint}/{n_ecm_eval} cointegrated; "
    f"{n_ec_neg_sig} sig-negative EC coef (informational only)"
)
print(
    f"POWER @coint,T30 : differenced max={diff_max:.3f} "
    f"(under-powered={DIFFERENCED_UNDERPOWERED_COINT_T30}); "
    f"TY max={lev_max:.3f} (well-powered={LEVELS_WELLPOWERED_COINT_T30})"
)
print(f"TY SIZE by T     : {ty_size_line}")
print(
    f"HONESTY (a)      : short-T size-inflation flag = {SHORT_T_FLAG} "
    f"(directional-T median {dir_T_med_str} vs evaluable median {T_eval_median})"
)
print(
    f"HONESTY (b)      : TY @coint/T30 well-powered only for LARGE beta "
    f"(first clears 0.80 at beta={ty_min_beta_powered}); a levels NULL rules out a LARGE "
    "level-cascade, not a small one."
)

F1 LEVELS DIAGNOSTIC (PR3D) — verdict: TRUE  | does_not_flip_g5 = True
LEVELS mean_tier : HOLDS=False  directional=7/138 (dominant quality_leads)  panel q->v=0.07415 v->q=0.8996
LEVELS rct_share : HOLDS=False  directional=0/138  panel q->v=0.8527 v->q=0.6518
DIFFERENCED ctrl : mean_tier 0/138 (panel 0.2771/0.7393) — reproduces locked F1 null = True
ECM (mean_tier)  : 50/128 cointegrated; 27 sig-negative EC coef (informational only)
POWER @coint,T30 : differenced max=0.382 (under-powered=False); TY max=0.884 (well-powered=True)
TY SIZE by T     : T8=0.00  T15=0.34  T22=0.14  T30=0.10  T40=0.07
HONESTY (a)      : short-T size-inflation flag = True (directional-T median 29 vs evaluable median 31)
HONESTY (b)      : TY @coint/T30 well-powered only for LARGE beta (first clears 0.80 at beta=2.0); a levels NULL rules out a LARGE level-cascade, not a small one.
